In [ ]:
import pandas as pd 
from pathlib import Path 
import os
import json 
import plotly.express as px
import numpy as np
import plotly.express as px

In [ ]:
PATH_RAW_DATA = Path(os.getcwd()) / "raw_data_trentino" 
assert PATH_RAW_DATA.exists()
vodafone_file = PATH_RAW_DATA / "vodafone_attendences.csv"

PATH_PROCESSED = Path(os.getcwd()) / "processed_data_trentino"
assert PATH_PROCESSED.exists()
phen_presenze = PATH_PROCESSED / "phen_presenze.parquet"

PATH_MAPPING = PATH_RAW_DATA / "mapping"

presenze_trentino_ispat = PATH_RAW_DATA / "presenze_Trentino_ISPAT.csv"
presenze_trentino_ispat_xalb = PATH_RAW_DATA / "presenze_Trentino_ISPAT_alb_xalb.csv"

In [ ]:
presenze = pd.read_parquet(phen_presenze)
vodafone_presenze = pd.read_csv(vodafone_file)

with open(
    PATH_RAW_DATA / "TRENTINO-comuni_Vodafone_2023.geojson",
    "r",
    encoding="utf-8",
) as f:
    geojson_comuni_json_data = json.load(f)

with open(PATH_MAPPING / "vodafone_Trento.json") as f:
    json_vodafone = json.load(f)

In [ ]:
presenze

In [ ]:
location_map = {
        feature["properties"]["id"]: feature["properties"]["name"].upper()
        for feature in geojson_comuni_json_data["features"]
    }

vodafone_presenze["comune"] = vodafone_presenze["locId"].map(
        location_map
    )

vodafone_presenze["ID_COMUNE"] = vodafone_presenze["comune"].map(
        json_vodafone
)

mask = vodafone_presenze["comune"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])

vodafone_presenze.loc[mask, "comune"] = "SAN GIOVANNI DI FASSA"
vodafone_presenze.loc[mask, "ID_COMUNE"] = [[22250]] * mask.sum()


vodafone_presenze_comuni = vodafone_presenze[
        (vodafone_presenze["locType"] == "TN_MKT_AL_3")]

In [ ]:
vodafone_presenze[vodafone_presenze['ID_COMUNE'].isna()].locId.unique()

In [ ]:
vodafone_presenze = vodafone_presenze.dropna(subset=['ID_COMUNE'])

In [ ]:
df_aggregato = vodafone_presenze_comuni.groupby(['date', 'comune', 'userProfile'])['value'].sum().reset_index()
df_aggregato

In [ ]:
commuter = df_aggregato[df_aggregato['userProfile'] == 'COMMUTER'] 
pivot_table =  commuter.pivot(index='date', columns='comune', values='value') 
corr_matrix = pivot_table.corr()  
corr_matrix

In [ ]:
def random_rgb():
    import secrets
    r = secrets.randbelow(256)
    g = secrets.randbelow(256)
    b = secrets.randbelow(256)
    return (r, g, b)

print(random_rgb())


In [ ]:
presenze['DATA'] = pd.to_datetime(presenze['DATA'])
presenze['MESE'] = presenze['DATA'].dt.to_period("M")
df_grouped = presenze.groupby(["MESE", "LOCATION", "ID_COMUNE"])['presenze_apt_mese_distributed'].sum().reset_index()

with open(PATH_MAPPING / "map_comuni_into_apt.json") as f:
    json_apt = json.load(f)
def atp_to_comuni():
    id_to_apt = {}
    for apt, id_list in json_apt.items():
        for id_comune in id_list:
            # Trasformiamo in int e poi in stringa a 6 cifre: "22001" -> "022001"
            id_6_cifre = str(int(id_comune)).zfill(6)
            id_to_apt[id_6_cifre] = apt
    return id_to_apt


df_grouped['ID_COMUNE'] = df_grouped['ID_COMUNE'].astype(str).str.zfill(6)
df_grouped['APT'] = df_grouped['ID_COMUNE'].map(atp_to_comuni())


In [ ]:
df_grouped.to_csv("grouped.csv")
df_grouped['Anno'] = df_grouped['MESE'].dt.year
df_grouped['Mese'] = df_grouped['MESE'].dt.month
df_grouped_pres_summed = df_grouped.groupby(["APT", "Anno", "Mese"])['presenze_apt_mese_distributed'].sum().reset_index().rename(columns = {'APT': 'Ambito'})
df_grouped_pres_summed

In [ ]:
pres_off_df = pd.read_csv(presenze_trentino_ispat)
pres_off_df = pres_off_df.sort_values("Ambito")
mg =  pres_off_df.merge(df_grouped_pres_summed, on =['Anno', 'Mese', 'Ambito'])
mg['diff'] = mg['Presenze'] - mg['presenze_apt_mese_distributed'] 
mg

In [ ]:
mg['diff'].describe()
mg[abs(mg['diff']) > 15]

In [ ]:
pres_off_df_noprov = pres_off_df[pres_off_df['Ambito'] != 'Provincia']
prov = pres_off_df[pres_off_df['Ambito'] == 'Provincia']
summed = pres_off_df_noprov.groupby(["Anno", "Mese"])['Presenze'].sum().reset_index()

In [ ]:
pres_off_xalb = pd.read_csv(presenze_trentino_ispat_xalb)
pres_off_xalb['mese'] = pres_off_xalb['Anno'].astype(str) + '-' + pres_off_xalb['Mese'].astype(str).str.zfill(2)
pres_off_xalb['mese'] = pd.to_datetime(pres_off_xalb['mese']).dt.to_period("M")
mgd = pres_off_xalb.merge(summed, on =['Anno' , 'Mese'])
mgd['diff'] = mgd['Presenze alberghi'] - mgd['Presenze']
pres_off_xalb

In [ ]:
vodafone_pres = df_aggregato.groupby(['date', 'comune'])['value'].sum().reset_index()
vodafone_pres[vodafone_pres['comune'] == 'ALA']

In [ ]:
# pres_off_xalb['ratio'] = pres_off_xalb['Presenze alberghi'] / pres_off_xalb['Presenze extra-alberghi']
# pres_off_xalb, pres_off_df, vodafone_pres

In [ ]:
vodafone_pres['date'] = pd.to_datetime(vodafone_pres['date'])
vodafone_pres['mese']= vodafone_pres['date'].dt.to_period("M")
vodafone_pres_month = vodafone_pres.groupby(["mese", "comune"])['value'].sum().reset_index()
vodafone_pres_monthprov = vodafone_pres.groupby(["mese"])['value'].sum().reset_index()

In [ ]:
## provincia in quel mese 
pres_off_xalb = pres_off_xalb.merge(vodafone_pres_monthprov, on = 'mese')
pres_off_xalb['ratio'] = pres_off_xalb ['Presenze alberghi'] / pres_off_xalb['value']
pres_off_xalb

In [ ]:
## apt in quel mese 
pres_off_df_noprov['ID'] = pres_off_df_noprov['Ambito'].map(json_apt)
vodafone_pres_month

In [ ]:
pres_off_df_noprov['mese'] = (
    pres_off_df_noprov['Anno'].astype(str) + '-' + 
    pres_off_df_noprov['Mese'].astype(str).str.zfill(2)
)

presenze['ID_NORM'] = presenze['ID_COMUNE'].astype(str).str.strip().str.lstrip('0')

df_esploso = pres_off_df_noprov[['Ambito', 'mese', 'ID']].explode('ID')
df_esploso = df_esploso.dropna(subset=['ID'])
pres_off_df_noprov

In [ ]:
comune_to_id = dict(zip(presenze['LOCATION'], presenze['ID_NORM']))


# === STEP 3: Normalizzazione del DF Vodafone ===
# Riconduciamo anche i comuni del DF Vodafone ai loro ID normalizzati
# (Sostituisci 'vodafone_pres_month' con il nome reale del tuo DataFrame di Vodafone)
vodafone_pres_month['ID_NORM'] = vodafone_pres_month['comune'].map(comune_to_id)


# === STEP 4: Explode e Merge ===
# 1. Esplodiamo la colonna 'ID' (le liste diventano righe singole temporanee)
df_esploso = pres_off_df_noprov[['Ambito', 'mese', 'ID']].explode('ID')
df_esploso = df_esploso.dropna(subset=['ID'])

# 2. Normalizziamo gli ID esplosi
df_esploso['ID_NORM'] = df_esploso['ID'].astype(str).str.strip().str.lstrip('0')
df_esploso['mese'] = pd.to_datetime(df_esploso['mese']).dt.to_period("M")
# 3. Facciamo il merge riga per riga usando Mese e ID_NORM come chiavi
merged = df_esploso.merge(
    vodafone_pres_month,
    left_on=['mese', 'ID_NORM'],
    right_on=['mese', 'ID_NORM'],
    how='left'
)
df_esploso[df_esploso['ID'] != df_esploso['ID_NORM']]
df_completo_apt = df_esploso.merge(vodafone_pres_month, on = ['mese', "ID_NORM"]).rename(columns= {"value" : "presenze_vodafone_somma"})

In [ ]:
df_grouped_apt = df_completo_apt.groupby(["Ambito", "mese"])['presenze_vodafone_somma'].sum().reset_index()
pres_off_df_noprov['mese'] = pres_off_df_noprov['Anno'].astype(str) + '-' + pres_off_df_noprov['Mese'].astype(str).str.zfill(2)
pres_off_df_noprov['mese'] = pd.to_datetime(pres_off_df_noprov['mese']).dt.to_period("M")
mgdt = pres_off_df_noprov.merge(df_grouped_apt, on = ['mese', 'Ambito'], how = 'outer').dropna()
mgdt['ratio'] = mgdt ['Presenze'] / mgdt ['presenze_vodafone_somma'] 
mgdt

In [ ]:
XALB = pres_off_xalb[['mese', 'ratio']]
APT = mgdt[['Ambito', 'mese', 'ratio']]
APT

In [ ]:
confronto = APT.merge(XALB, on='mese', suffixes=('_ambito', '_prov'))
confronto['scarto_assoluto'] = confronto['ratio_ambito'] - confronto['ratio_prov']
confronto['scarto_relativo'] = confronto['scarto_assoluto'] / confronto['ratio_prov']

confronto = confronto.sort_values('scarto_relativo', key=abs, ascending=False)
confronto


In [ ]:
confronto['mese_str'] = confronto['mese'].astype(str)

fig = px.line(
    confronto.sort_values('mese'),
    x='mese_str', y='ratio_ambito', color='Ambito',
    title='Rapporto Presenze alberghiere ISPAT / Vodafone, per ambito nel tempo'
)
# ratio provinciale come riferimento
prov_ref = XALB.sort_values('mese')
fig.add_scatter(x=prov_ref['mese'].astype(str), y=prov_ref['ratio'],
                mode='lines', name='Media provincia', line=dict(color='black', dash='dash'))
fig.show()